1. Loading job profiles from a parquet file
2. Converting each row into a clean profile string (skipped if `profile_text` already exists)
3. Embedding profile text (resumable via memmap checkpoint)
4. Creating Elasticsearch `job_profiles` index and bulk-uploading profiles + embeddings
5. Verifying document count

In [1]:
import sys
from pathlib import Path

import pyarrow.parquet as pq

sys.path.insert(0, str(Path('..').resolve()))

from core.config import EmbeddingConfig, ElasticsearchConfig
from core.embeddings import embed_all
from core.elasticsearch import bulk_index, create_index, get_es_client, job_actions
from core.indices import JOB_INDEX, job_mappings
from core.profiles import row_to_job_profile

DATASET_PATH = Path('../data/job_profiles.parquet')
ID_COLUMN = 'id'
MAX_ROWS = None  # set an int for a smaller test run
EMBEDDINGS_PATH = Path('../data/job_embeddings.npy')
PROGRESS_PATH = Path('../data/job_embed_progress.json')
ES_INDEX = JOB_INDEX

embed_config = EmbeddingConfig.from_env()
es_config = ElasticsearchConfig.from_env()

In [ ]:
# read parquet file
df = pq.read_table(DATASET_PATH).to_pandas()

print(f'Loaded rows: {len(df):,}')
print('Columns:', list(df.columns))

Loaded rows: 141,897
Columns: ['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', '__index_level_0__']


In [ ]:
# limit rows if specified
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(f'Kept rows: {len(df):,}')

Kept rows: 141,897


In [ ]:
# convert each row into a profile string if not already done
if 'profile_text' not in df.columns:
    df['profile_text'] = df.apply(row_to_job_profile, axis=1)

print(f'Profile texts ready: {len(df):,}')
print(df['profile_text'].iloc[0][:500])

Profile texts ready: 141,897
Position: 10 + Blockchain Nodes / Masternodes to set up
Description: *Requirements* We're looking for a long term collaboration with someone that has an experience in crypto, masternodes, nodes, validators etc. We need to set up: Kyber Network Nebulas SecretNetwork Tron Aion DeFiChain EOS TomoChain Elrond IRISnet IoTeX Terra ChainX Thorchain Succesful candidates will have an opportunity to get more jobs and long term collaboration.
Company: MyCointainer
Experience required: 2y
Primary keyword: S


In [5]:
# embed job profiles
job_embeddings = embed_all(
    df['profile_text'].tolist(),
    EMBEDDINGS_PATH,
    PROGRESS_PATH,
    embed_config,
)

Embedding 141,897 profiles via https://qwen-embed-runpod.boliu.info/v1/embeddings
Model: Qwen/Qwen3-Embedding-0.6B  |  batch size: 32  |  workers: 4
Resuming from offset: 0  |  pending batches: 4,435


API batches:   0%|          | 0/4435 [00:00<?, ?batch/s]

Done. Embeddings shape: (141897, 1024)


In [6]:
# create es index and overwrite if exists
es = get_es_client(es_config)
info = es.info()
print(f"Connected to Elasticsearch {info['version']['number']}")

create_index(es, ES_INDEX, job_mappings(embed_config))

/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/elasticsearch/_sync/client/__init__.py:328: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/

Connected to Elasticsearch 9.4.2
Created index: job_profiles


In [7]:
# bulk upload to es
bulk_index(
    es,
    ES_INDEX,
    job_actions(df, job_embeddings, index=ES_INDEX, id_column=ID_COLUMN),
)

/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warning

Indexed documents: 141,897


(141897, [])

In [8]:
# verify document count
doc_count = es.count(index=ES_INDEX)['count']
print(f'Documents in {ES_INDEX}: {doc_count:,} (expected {len(df):,})')

Documents in job_profiles: 141,897 (expected 141,897)


/home/zach/Projects/semantic-search/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
